## Ensembles: Random Forest, Gradient Boosting en XGBoost

**Wat is een ensemble?**
Een ensemble combineert de voorspellingen van meerdere modellen tot één voorspelling. Eén decision tree maakt snel fouten en overfit makkelijk, maar als je veel verschillende bomen combineert, middelen de fouten elkaar uit.

Er zijn twee hoofdtechnieken:
- **Bagging** (Random Forest): veel bomen worden onafhankelijk van elkaar getraind, elk op een willekeurige steekproef van de data (bootstrap) en met een willekeurige selectie van features. De voorspelling is de meerderheid van de stemmen. Dit verlaagt vooral de **variance**.
- **Boosting** (Gradient Boosting, XGBoost): bomen worden na elkaar getraind. Elke nieuwe boom probeert de fouten van de vorige bomen te corrigeren. Dit verlaagt vooral de **bias**.

**Waarom ensembles voor deze dataset?**
Decision trees kunnen niet-lineaire verbanden en interacties oppikken, zoals het effect van een hoge `avg_glucose_level` of de combinatie van hoge leeftijd en hoge bloeddruk. Logistic regression kan dit niet.

Ensembles op basis van decision trees hebben geen standaardisatie nodig: een boom splitst op één feature tegelijk, waardoor de schaal geen invloed heeft.

### Random Forest: baseline

We beginnen met een Random Forest met standaardinstellingen en 300 bomen. We evalueren op dezelfde manier als bij logistic regression: 5-fold `StratifiedKFold`.

In [1]:
"""Ensembles voor de Kaggle-competitie Predicting stroke."""
import sys

sys.path.append("..")  # zodat de map src gevonden wordt

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.preprocessing import StrokeData

data = StrokeData()
X, y = data.get_xy()
X_test = data.get_test()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
}

rf_default = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

scores = cross_validate(rf_default, X, y, cv=cv, scoring=scoring)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

precision: 0.000
recall: 0.000
f1: 0.000


**Observatie:** net als de baseline van logistic regression voorspelt het Random Forest voor geen enkele patiënt een herseninfarct. Bijna alle bladeren van de bomen bevatten vooral patiënten zonder herseninfarct, waardoor de meerderheid van de 300 bomen altijd "geen herseninfarct" stemt. Een krachtiger model lost de onbalans dus niet vanzelf op.

### Class weights

Bij logistic regression loste `class_weight="balanced"` het probleem direct op. We proberen hetzelfde bij het Random Forest: een patiënt met een herseninfarct telt dan zwaarder mee bij het kiezen van de splits en bij het stemmen in de bladeren.

In [2]:
rf_balanced = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)

scores = cross_validate(rf_balanced, X, y, cv=cv, scoring=scoring)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

precision: 0.080
recall: 0.021
f1: 0.033


**Observatie:** anders dan bij logistic regression helpt `class_weight="balanced"` hier nauwelijks: de recall is slechts 0,021. De oorzaak is overfitting. Standaard groeit elke boom door tot alle bladeren zuiver zijn (alleen patiënten van één klasse). De bomen leren de trainingsdata dan uit het hoofd. In een zuiver blad valt er niets te wegen, dus de class weights hebben geen effect. Nieuwe patiënten met een herseninfarct komen vrijwel altijd terecht in een blad met "geen herseninfarct".

### Overfitting beperken met `min_samples_leaf`

Met `min_samples_leaf` bepalen we hoeveel patiënten er minimaal in een blad moeten zitten. Een hogere waarde maakt de bomen eenvoudiger: de bladeren blijven gemengd, waardoor de class weights effect hebben. We vergelijken een aantal waarden.

In [3]:
rows = []
for leaf in [1, 5, 20, 50, 100]:
    rf = RandomForestClassifier(
        n_estimators=300, min_samples_leaf=leaf, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )
    scores = cross_validate(rf, X, y, cv=cv, scoring=scoring)
    rows.append({
        "min_samples_leaf": leaf,
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
    })

pd.DataFrame(rows).round(3)

,min_samples_leaf,precision,recall,f1
0,1,0.080,0.021,0.033
1,5,0.073,0.306,0.118
2,20,0.060,0.681,0.110
3,50,0.052,0.772,0.097
4,100,0.048,0.816,0.091


**Observatie:** hoe groter `min_samples_leaf`, hoe hoger de recall (van 0,021 naar 0,816) en hoe lager de precision. De bladeren blijven gemengd, waardoor de class weights nu wel effect hebben. Dit is de bias-variance tradeoff: kleine bladeren geven complexe bomen die overfitten (hoge variance), grote bladeren geven eenvoudigere bomen die beter generaliseren. Bij de standaard threshold van 0,5 is de F1-score het hoogst bij `min_samples_leaf = 5` (0,118). Deze vergelijking is echter niet eerlijk: elk model heeft een andere optimale threshold.

### Threshold per model

Net als bij logistic regression zoeken we per model de threshold met de hoogste F1-score. Pas daarna kunnen we de waarden van `min_samples_leaf` eerlijk vergelijken.

In [4]:
import numpy as np
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import cross_val_predict

thresholds = np.arange(0.05, 0.96, 0.05)

rows = []
for leaf in [5, 20, 50, 100]:
    rf = RandomForestClassifier(
        n_estimators=300, min_samples_leaf=leaf, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )
    y_proba = cross_val_predict(rf, X, y, cv=cv, method="predict_proba")[:, 1]
    f1_scores = [f1_score(y, (y_proba >= t).astype(int)) for t in thresholds]
    best_threshold = thresholds[int(np.argmax(f1_scores))]
    y_pred = (y_proba >= best_threshold).astype(int)
    rows.append({
        "min_samples_leaf": leaf,
        "beste_threshold": round(best_threshold, 2),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1": max(f1_scores),
    })

pd.DataFrame(rows).round(3)

,min_samples_leaf,beste_threshold,precision,recall,f1
0,5,0.45,0.072,0.397,0.121
1,20,0.65,0.076,0.484,0.132
2,50,0.75,0.086,0.342,0.137
3,100,0.75,0.084,0.335,0.134


**Observatie:** met een eigen threshold per model verandert de volgorde volledig. `min_samples_leaf = 5` scoorde bij threshold 0,5 het best, maar is nu het slechtst (0,121). De waarden 20, 50 en 100 liggen dicht bij elkaar (0,132–0,137). Dit laat zien dat modellen alleen eerlijk te vergelijken zijn als de threshold per model is afgestemd. We kiezen `min_samples_leaf = 50`: de hoogste F1-score (0,137) met eenvoudige bomen die minder snel overfitten. Geen van de modellen haalt de 0,143 van logistic regression.

### Eerlijke evaluatie van het Random Forest

In de tabel hierboven is de threshold gekozen op dezelfde voorspellingen waarop de score is gemeten. Net als bij logistic regression gebruiken we daarom `TunedThresholdClassifierCV`, zodat de threshold binnen elke fold alleen op de trainingsdata wordt gekozen.

In [5]:
from sklearn.model_selection import TunedThresholdClassifierCV

rf_best = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=50, class_weight="balanced",
    random_state=42, n_jobs=-1,
)
rf_tuned = TunedThresholdClassifierCV(rf_best, scoring="f1", cv=5, random_state=42)

scores = cross_validate(rf_tuned, X, y, cv=cv, scoring=scoring, return_estimator=True)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

print("Gekozen thresholds:", [round(m.best_threshold_, 2) for m in scores["estimator"]])

precision: 0.081
recall: 0.412
f1: 0.135
Gekozen thresholds: [np.float64(0.72), np.float64(0.75), np.float64(0.7), np.float64(0.74), np.float64(0.73)]


**Observatie:** met een eerlijk gekozen threshold haalt het Random Forest een F1-score van 0,135 (precision 0,081, recall 0,412). De gekozen thresholds liggen in alle folds tussen 0,70 en 0,75, dus de keuze is stabiel. Logistic regression scoort iets hoger (0,143) bij dezelfde recall. Het verschil is klein en ligt in de buurt van de spreiding tussen de folds. Een veel complexer model levert dus geen winst op. Dit wijst erop dat niet het model de beperkende factor is, maar de informatie in de features: veel patiënten met dezelfde risicofactoren krijgen geen herseninfarct.

### Gradient Boosting

Bij boosting worden de bomen na elkaar getraind. Elke nieuwe boom is klein (standaard diepte 3) en leert van de fouten van alle vorige bomen samen. De `learning_rate` bepaalt hoe zwaar elke nieuwe boom meetelt.

`GradientBoostingClassifier` heeft geen `class_weight`. We vangen de onbalans daarom alleen op met de threshold. We vergelijken de standaard threshold (0,5) met een getunede threshold.

In [6]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

scores = cross_validate(gb, X, y, cv=cv, scoring=scoring, n_jobs=-1)
print("Threshold 0,5")
for metric in scoring:
    print(f"  {metric}: {scores[f'test_{metric}'].mean():.3f}")

gb_tuned = TunedThresholdClassifierCV(gb, scoring="f1", cv=5, random_state=42)
scores = cross_validate(gb_tuned, X, y, cv=cv, scoring=scoring,
                        return_estimator=True, n_jobs=-1)
print("Getunede threshold")
for metric in scoring:
    print(f"  {metric}: {scores[f'test_{metric}'].mean():.3f}")
print("Gekozen thresholds:", [round(m.best_threshold_, 2) for m in scores["estimator"]])

Threshold 0,5
  precision: 0.215
  recall: 0.008
  f1: 0.015
Getunede threshold
  precision: 0.079
  recall: 0.474
  f1: 0.135
Gekozen thresholds: [np.float64(0.05), np.float64(0.05), np.float64(0.05), np.float64(0.05), np.float64(0.04)]


**Observatie:** bij threshold 0,5 vindt Gradient Boosting vrijwel geen patiënten met een herseninfarct (recall 0,008). Met een getunede threshold is de F1-score 0,135, gelijk aan het Random Forest, met een iets hogere recall (0,474).

De gekozen threshold is ongeveer 0,05, veel lager dan bij het Random Forest (0,73) en logistic regression (0,80). Zonder class weights voorspelt het model realistische kansen: gemiddeld heeft 1,54% een herseninfarct, dus een patiënt met een kans van 5% heeft al een ruim drie keer zo hoog risico. Met `class_weight="balanced"` worden de kansen kunstmatig verhoogd en is een hoge threshold nodig. Beide aanpakken leiden tot vrijwel dezelfde score: class weights en een lage threshold zijn twee manieren om hetzelfde te bereiken.

### XGBoost

XGBoost is een snellere variant van gradient boosting met ingebouwde regularisatie. De standaardinstellingen zijn vrij agressief: diepe bomen (`max_depth=6`) en een hoge `learning_rate` (0,3). Bij het Random Forest zagen we dat complexe bomen overfitten op deze dataset. We vergelijken daarom de standaardinstellingen met een eenvoudiger model: ondiepe bomen (`max_depth=3`), een lage `learning_rate` (0,05) en meer bomen (200).

In [7]:
from xgboost import XGBClassifier

xgb_models = {
    "standaard": XGBClassifier(random_state=42, n_jobs=-1),
    "eenvoudiger": XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        random_state=42, n_jobs=-1,
    ),
}

rows = []
for name, model in xgb_models.items():
    tuned = TunedThresholdClassifierCV(model, scoring="f1", cv=5, random_state=42)
    scores = cross_validate(tuned, X, y, cv=cv, scoring=scoring)
    rows.append({
        "model": name,
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
    })

pd.DataFrame(rows).round(3)

,model,precision,recall,f1
0,standaard,0.058,0.308,0.096
1,eenvoudiger,0.084,0.422,0.139


**Observatie:** XGBoost met standaardinstellingen scoort duidelijk slechter (F1 0,096) dan het eenvoudigere model (0,139). Net als bij het Random Forest overfitten diepe bomen op deze dataset: met slechts 517 patiënten met een herseninfarct leren complexe bomen toevallige patronen uit de trainingsdata. Ondiepe bomen met een lage `learning_rate` generaliseren beter. Het eenvoudigere XGBoost-model is het beste ensemble tot nu toe, maar blijft net onder logistic regression (0,143).

## Zelf samengesteld ensemble

We combineren drie verschillende modellen met een `VotingClassifier`: logistic regression, Random Forest en XGBoost. Bij **soft voting** wordt het gemiddelde genomen van de voorspelde kansen van de drie modellen.

Een ensemble werkt het best als de modellen verschillende soorten fouten maken. Logistic regression legt lineaire verbanden, de boommodellen kunnen niet-lineaire verbanden en interacties oppikken.

Om de kansen te kunnen middelen moeten ze op dezelfde schaal liggen. Daarom gebruiken alle drie de modellen class weights. Bij XGBoost heet dit `scale_pos_weight`: de verhouding tussen het aantal patiënten zonder en met een herseninfarct.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.preprocessing import NUMERIC_COLS

logreg = Pipeline([
    ("pre", ColumnTransformer(
        [("scale", StandardScaler(), NUMERIC_COLS)], remainder="passthrough"
    )),
    ("clf", LogisticRegression(max_iter=1000, random_state=42,
                               class_weight="balanced")),
])

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
xgb_best = XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1,
)

voting = VotingClassifier(
    [("logreg", logreg), ("rf", rf_best), ("xgb", xgb_best)],
    voting="soft",
)
voting_tuned = TunedThresholdClassifierCV(voting, scoring="f1", cv=5, random_state=42)

scores = cross_validate(voting_tuned, X, y, cv=cv, scoring=scoring, return_estimator=True)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")
print("Gekozen thresholds:", [round(m.best_threshold_, 2) for m in scores["estimator"]])
print("scale_pos_weight:", round(scale_pos_weight, 1))

precision: 0.088
recall: 0.410
f1: 0.142
Gekozen thresholds: [np.float64(0.78), np.float64(0.78), np.float64(0.75), np.float64(0.74), np.float64(0.81)]
scale_pos_weight: 63.9


**Observatie:** het ensemble haalt een F1-score van 0,142 (precision 0,088, recall 0,410). Dat is beter dan het Random Forest (0,135) en XGBoost (0,139) afzonderlijk, maar gelijk aan logistic regression alleen (0,143). De thresholds liggen tussen 0,74 en 0,81 en zijn dus stabiel.

Een ensemble levert winst op als de modellen verschillende fouten maken. Hier maken de drie modellen grotendeels dezelfde fouten: oudere patiënten met risicofactoren die toch geen herseninfarct krijgen. Het middelen van de kansen lost dat niet op. Alle modellen lopen tegen hetzelfde plafond aan (F1 ≈ 0,14). De beperking zit in de data, niet in het model.